<a href="https://colab.research.google.com/github/balajigund/Chat-with-pdf-system/blob/main/chat_with_pdf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
!pip install PyPDF2 sentence-transformers faiss-cpu

from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from google.colab import files

# Upload PDF
uploaded = files.upload()

# Read PDF
pdf_file = list(uploaded.keys())[0]
reader = PdfReader(pdf_file)

text = []
for page in reader.pages:
    content = page.extract_text()
    if content:
        text.append(content)

# Split text into chunks
chunks = []
for t in text:
    chunks.extend([t[i:i+300] for i in range(0, len(t), 300)])

print(f"📄 Loaded {len(chunks)} text chunks from PDF")

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings
embeddings = model.encode(chunks)

# Create FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

# Ask question
while True:
    query = input("\n💬 Ask a question (or type 'exit'): ")
    if query.lower() == "exit":
        break

    query_vec = model.encode([query])
    D, I = index.search(np.array(query_vec), k=1)

    print("\n📌 Answer from document:\n", chunks[I[0][0]])